# TF Expression → Gene Expression: Linear Regression with Causal Constraints

Can we predict how gene expression changes under TF knockdowns using only TF expression levels as predictors?

**Setting:** CRISPR knockdown screen. Each cell has one TF knocked down (or is a control). We use the measured TF expression (log-CP10K) as features.

- **Train:** All TF-knockdown cells + controls (model sees how KD changes TF expression → downstream effects)
- **Val:** Non-TF perturbation cells (held out — tests generalization)
- **Likelihood:** Negative Binomial on raw counts

**Three models compared:**
1. **Baseline** — predict control mean expression (no learning of TF effects)
2. **Unconstrained** — all TF expression → all genes (dense linear)
3. **Causal** — only known RegulonDB TF→gene edges (sparse linear)

In [ ]:
import sys

sys.path.insert(0, "/workspace/src")

import numpy as np
import pandas as pd
import scanpy as sc
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
from flax.linen.initializers import glorot_normal, zeros
from flax.training import train_state
from numpyro.distributions import NegativeBinomial2
from scipy import sparse
import matplotlib.pyplot as plt
import plotnine as gg
from essential.utils import PLOTNINE_DEFAULT_THEME_2
from tqdm import tqdm
from essential.data import load_regulondb_full
import plotnine as gg

In [ ]:
ADATA_PATH = "/workspace/data/de122_lce75/adata_de122_lce75_merged.h5ad"
PERT_COL = "target"
CTRL_KEY = "nontargeting"
EXPERIMENT_SUBSET = "lce75"
MIN_LIB = 1e3

N_EPOCHS = 50
BATCH_SIZE = 512
LR = 1e-3

## 1. Data

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
adata.obs["_lib"] = np.asarray(adata.layers["reads"].sum(1)).ravel()
adata = adata[
    (adata.obs["_lib"] > MIN_LIB)
    & (adata.obs["experiment"] == EXPERIMENT_SUBSET)
    & adata.obs[PERT_COL].notna()
].copy()
adata.var_names = adata.var_names.str.lower()
adata.obs[PERT_COL] = adata.obs[PERT_COL].str.lower()

raw = adata.layers["reads"]
raw = raw.toarray() if sparse.issparse(raw) else np.asarray(raw, np.float32)
lib_col = raw.sum(1, keepdims=True)
adata.layers["counts"] = raw.astype(np.float32)
adata.layers["lcp10k"] = np.log1p(raw / (lib_col + 1e-6) * 1e4).astype(np.float32)
adata.X = adata.layers["counts"]
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes")

In [ ]:
ref_db = load_regulondb_full()
ref_db = ref_db[ref_db["ri_type"].str.startswith("TF")].copy()
all_tfs = set(ref_db["regulator_gene"].str.lower().unique())

perts = adata.obs[PERT_COL]
obs_targets = set(perts.unique()) - {CTRL_KEY}
tf_perts = obs_targets & all_tfs
non_tf_perts = obs_targets - all_tfs

adata_train = adata[perts.isin(tf_perts) | (perts == CTRL_KEY)].copy()
adata_val = adata[perts.isin(non_tf_perts)].copy()

print(
    f"Train: {adata_train.n_obs:,} cells  ({len(tf_perts)} TF perturbations + controls)"
)
print(f"Val:   {adata_val.n_obs:,} cells  ({len(non_tf_perts)} non-TF perturbations)")

In [ ]:
var_names = list(adata.var_names)
gene_idx = {g: i for i, g in enumerate(var_names)}
tf_genes = sorted(g for g in all_tfs if g in gene_idx)
tf_idx_map = {g: i for i, g in enumerate(tf_genes)}
tf_cols = np.array([gene_idx[g] for g in tf_genes])
n_genes, n_tfs = adata.n_vars, len(tf_genes)

# Amask_tf[i, k] = 1  <=>  TF k regulates gene i  (RegulonDB)
Amask_tf = np.zeros((n_genes, n_tfs), dtype=np.float32)
for _, row in ref_db.iterrows():
    t, r = row["target_gene"].lower(), row["regulator_gene"].lower()
    if t in gene_idx and r in tf_idx_map:
        Amask_tf[gene_idx[t], tf_idx_map[r]] = 1.0

print(
    f"{n_tfs} TF genes  |  {int(Amask_tf.sum()):,} RegulonDB edges  |  "
    f"{int((Amask_tf.sum(1) > 0).sum())} genes with >= 1 regulator"
)


def get_arrays(a):
    lcp = np.asarray(a.layers["lcp10k"], dtype=np.float32)
    raw = np.asarray(a.layers["counts"], dtype=np.float32)
    return lcp, raw


lcp_train, raw_train = get_arrays(adata_train)
lcp_val, raw_val = get_arrays(adata_val)

ctrl_mask = np.asarray(adata_train.obs[PERT_COL] == CTRL_KEY)
tf_mu = lcp_train[ctrl_mask][:, tf_cols].mean(0)
tf_sigma = lcp_train[ctrl_mask][:, tf_cols].std(0)
tf_sigma = np.where(tf_sigma > 1e-3, tf_sigma, 1.0)

Xtrain = (lcp_train[:, tf_cols] - tf_mu) / tf_sigma
Xval = (lcp_val[:, tf_cols] - tf_mu) / tf_sigma
lib_train = raw_train.sum(1)
lib_val = raw_val.sum(1)

ctrl_lcp_mean = lcp_train[ctrl_mask].mean(0)

## 2. Model

For each gene $g$, we predict the log-CP10K mean:
$$\hat{\mu}_g^{\text{lcp10k}} = \mathbf{w}_g^\top \tilde{\mathbf{x}}_{\text{TF}} + b_g$$

then convert to raw-count scale for the NB likelihood:
$$\mu_g = \max\!\left(e^{\hat{\mu}_g} - 1,\, \varepsilon\right) \cdot \frac{L}{10^4}$$

**Causal constraint:** $\mathbf{w}_g$ is masked to zero for TFs not connected to gene $g$ in RegulonDB.

In [ ]:
class TFLinearNB(nn.Module):
    n_genes: int
    n_tfs: int
    x_mean: jnp.ndarray  # (n_genes,) frozen control log-CP10K mean
    Amask_tf: jnp.ndarray | None = None  # (n_genes, n_tfs); None = unconstrained

    @nn.compact
    def __call__(self, x_tf, y_raw, lib):
        """x_tf: (B, n_tfs) normalised  |  y_raw: (B, n_genes) counts  |  lib: (B,)"""
        W = self.param("W", glorot_normal(), (self.n_genes, self.n_tfs))
        b = self.param("b", zeros, (self.n_genes,))
        overdispersion_ = self.param("overdispersion_", zeros, (self.n_genes,))

        mask = (
            jax.lax.stop_gradient(self.Amask_tf) if self.Amask_tf is not None else 1.0
        )
        x_mean = jax.lax.stop_gradient(jnp.array(self.x_mean))
        lcp_pred = x_mean + x_tf @ (W * mask).T + b  # residual from control mean
        lib_scale = lib[:, None] / 1e4
        mean = jnp.maximum(jnp.expm1(lcp_pred) * lib_scale, 1e-8)
        conc = jnp.exp(overdispersion_)
        nll = -NegativeBinomial2(mean=mean, concentration=conc).log_prob(y_raw).mean()
        return {"loss": nll, "nll": nll}

## 3. Training

In [ ]:
def make_step(model):
    @jax.jit
    def step(state, x, y, lib):
        def loss_fn(params):
            out = model.apply({"params": params}, x, y, lib)
            return out["loss"], out

        (_, out), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
        return state.apply_gradients(grads=grads), out

    return step


def train(model, X, Y, lib, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR):
    key = jax.random.PRNGKey(0)
    params = model.init(key, jnp.array(X[:4]), jnp.array(Y[:4]), jnp.array(lib[:4]))[
        "params"
    ]
    state = train_state.TrainState.create(
        apply_fn=model.apply, params=params, tx=optax.adam(lr)
    )
    step = make_step(model)
    rng, history = np.random.default_rng(0), []
    n = len(X)
    for _ in tqdm(range(n_epochs)):
        idx = rng.permutation(n)
        losses = []
        for s in range(0, n - batch_size + 1, batch_size):
            b = idx[s : s + batch_size]
            state, out = step(
                state, jnp.array(X[b]), jnp.array(Y[b]), jnp.array(lib[b])
            )
            losses.append(float(out["nll"]))
        history.append(np.mean(losses))
    return state, history

In [ ]:
x_mean_frozen = jnp.array(ctrl_lcp_mean)

model_unc = TFLinearNB(
    n_genes=n_genes, n_tfs=n_tfs, x_mean=x_mean_frozen, Amask_tf=None
)
model_cau = TFLinearNB(
    n_genes=n_genes, n_tfs=n_tfs, x_mean=x_mean_frozen, Amask_tf=jnp.array(Amask_tf)
)

print("Training unconstrained...")
state_unc, hist_unc = train(model_unc, Xtrain, raw_train, lib_train)

print("Training causal (RegulonDB)...")
state_cau, hist_cau = train(model_cau, Xtrain, raw_train, lib_train)

In [ ]:
# ── Parameter diagnostics ────────────────────────────────────────────────────
for label, state in [("Unconstrained", state_unc), ("Causal", state_cau)]:
    p = state.params
    W = np.array(p["W"])
    b = np.array(p["b"])
    ovd = np.array(p["overdispersion_"])

    mask = Amask_tf if label == "Causal" else np.ones_like(Amask_tf)
    W_eff = W * mask

    preact_range = np.abs(Xval @ W_eff.T).max(0)  # max |W·x_tf| per gene across val set

    print(f"\n=== {label} ===")
    print(
        f"  W_eff   : min={W_eff.min():.3f}  max={W_eff.max():.3f}  std={W_eff.std():.4f}  "
        f"nonzero mean-max-abs={np.abs(W_eff).max(1).mean():.4f}"
    )
    print(f"  b       : min={b.min():.3f}  max={b.max():.3f}  std={b.std():.4f}")
    print(f"  overdispersion_: min={ovd.min():.3f}  max={ovd.max():.3f}")
    print(
        f"  lcp_pred at x_tf=0  (= ctrl_mean + b): "
        f"min={( ctrl_lcp_mean + b).min():.3f}  max={(ctrl_lcp_mean + b).max():.3f}  "
        f"std={(ctrl_lcp_mean + b).std():.3f}"
    )
    print(
        f"  max |W·x_tf| per gene (val): "
        f"mean={preact_range.mean():.3f}  p50={np.median(preact_range):.3f}  "
        f"p99={np.percentile(preact_range, 99):.3f}"
    )

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for row, (label, state) in enumerate(
    [("Unconstrained", state_unc), ("Causal", state_cau)]
):
    p = state.params
    W = np.array(p["W"])
    b = np.array(p["b"])
    mask = Amask_tf if label == "Causal" else np.ones_like(Amask_tf)
    W_eff = W * mask

    axes[row, 0].hist(W_eff[W_eff != 0].ravel(), bins=80, color=f"C{row}")
    axes[row, 0].set_title(f"{label}: W (nonzero entries)")
    axes[row, 0].set_xlabel("weight")

    axes[row, 1].hist(b, bins=60, color=f"C{row}")
    axes[row, 1].set_title(f"{label}: b (residual bias)")
    axes[row, 1].set_xlabel("bias")

    axes[row, 2].hist(ctrl_lcp_mean + b, bins=60, color=f"C{row}")
    axes[row, 2].set_title(f"{label}: lcp_pred at x_tf=0")
    axes[row, 2].set_xlabel("ctrl_mean + b")

    preact_range = np.abs(Xval @ W_eff.T).max(0)
    axes[row, 3].hist(preact_range, bins=60, color=f"C{row}")
    axes[row, 3].axvline(1.0, color="k", lw=0.8, ls="--", label="range=1")
    axes[row, 3].set_title(f"{label}: max |W·x_tf| per gene (val)")
    axes[row, 3].set_xlabel("covariate contribution range")
    axes[row, 3].legend()

plt.tight_layout()
plt.show()

In [ ]:
a = state_cau.params["W"] * jnp.array(Amask_tf)
a[a != 0]

## 4. Evaluation

In [ ]:
def eval_nll(params, amask_tf, x_mean, X, Y, lib, batch_size=512):
    n = len(X)
    lkl_sum, n_total = np.zeros(n_genes), 0
    mask = jnp.array(amask_tf) if amask_tf is not None else 1.0
    x_mean_j = jnp.array(x_mean)
    for s in range(0, n, batch_size):
        x_b = jnp.array(X[s : s + batch_size])
        y_b = jnp.array(Y[s : s + batch_size])
        l_b = jnp.array(lib[s : s + batch_size])
        lcp_pred = x_mean_j + x_b @ (params["W"] * mask).T + params["b"]
        mean = jnp.maximum(jnp.expm1(lcp_pred) * l_b[:, None] / 1e4, 1e-8)
        conc = jnp.exp(params["overdispersion_"])
        lkl_sum += np.asarray(
            NegativeBinomial2(mean=mean, concentration=conc).log_prob(y_b).sum(0)
        )
        n_total += len(x_b)
    return -lkl_sum / n_total


def eval_baseline_nll(log_conc, Y, lib, batch_size=512):
    n = len(Y)
    lkl_sum, n_total = np.zeros(n_genes), 0
    mu_ctrl = jnp.array(ctrl_lcp_mean)
    for s in range(0, n, batch_size):
        y_b = jnp.array(Y[s : s + batch_size])
        l_b = jnp.array(lib[s : s + batch_size])
        mean = jnp.maximum(jnp.expm1(mu_ctrl)[None, :] * l_b[:, None] / 1e4, 1e-8)
        conc = jnp.exp(jnp.array(log_conc))
        lkl_sum += np.asarray(
            NegativeBinomial2(mean=mean, concentration=conc).log_prob(y_b).sum(0)
        )
        n_total += len(y_b)
    return -lkl_sum / n_total


nll_unc = eval_nll(state_unc.params, None, ctrl_lcp_mean, Xval, raw_val, lib_val)
nll_cau = eval_nll(state_cau.params, Amask_tf, ctrl_lcp_mean, Xval, raw_val, lib_val)
nll_bl = eval_baseline_nll(
    np.array(state_unc.params["overdispersion_"]), raw_val, lib_val
)

# Scalar NLL across all observations and genes (for reporting / downstream use)
total_nll_bl = float(nll_bl.mean())
total_nll_unc = float(nll_unc.mean())
total_nll_cau = float(nll_cau.mean())

df_summary = pd.DataFrame(
    {
        "model": ["Baseline (ctrl mean)", "Unconstrained", "Causal (RegulonDB)"],
        "mean_nll": [total_nll_bl, total_nll_unc, total_nll_cau],
        "median_nll": [np.median(nll_bl), np.median(nll_unc), np.median(nll_cau)],
    }
)
print(df_summary.to_string(index=False))
print(f"\nScalar val NLL (mean over genes and cells):")
print(f"  Baseline:      {total_nll_bl:.4f}")
print(f"  Unconstrained: {total_nll_unc:.4f}")
print(f"  Causal:        {total_nll_cau:.4f}")

In [ ]:
# ── MLE overdispersion for the constant-mean baseline ─────────────────────────
# eval_baseline_nll above used state_unc.params["overdispersion_"], which was
# jointly trained with the unconstrained mean predictor — not with ctrl_lcp_mean.
# that model badly overfit (val NLL 2.54 vs 1.62), so its concentration params
# are miscalibrated for a constant-mean predictor.
#
# correct approach: MLE of conc holding mean fixed at expm1(ctrl_lcp_mean)*lib/1e4.

nll_bl_wrong = nll_bl.copy()  # save for comparison
total_nll_bl_wrong = total_nll_bl


def fit_baseline_overdispersion(Y, lib, n_steps=400, lr=5e-3, batch_size=BATCH_SIZE):
    """Adam MLE for per-gene NB concentration, baseline mean fixed."""
    mu_bl = jnp.maximum(jnp.expm1(jnp.array(ctrl_lcp_mean)), 1e-8)
    log_conc = jnp.zeros(n_genes)

    @jax.jit
    def loss_and_grad(lc, y_b, l_b):
        mean_b = mu_bl[None, :] * l_b[:, None] / 1e4

        def nll(lc):
            return (
                -NegativeBinomial2(mean=mean_b, concentration=jnp.exp(lc))
                .log_prob(y_b)
                .mean()
            )

        return jax.value_and_grad(nll)(lc)

    opt = optax.adam(lr)
    opt_state = opt.init(log_conc)
    rng = np.random.default_rng(0)
    n = len(Y)

    for _ in range(n_steps):
        idx = rng.integers(0, n, size=batch_size)
        loss, g = loss_and_grad(log_conc, jnp.array(Y[idx]), jnp.array(lib[idx]))
        updates, opt_state = opt.update(g, opt_state)
        log_conc = optax.apply_updates(log_conc, updates)

    return np.array(log_conc)


print("fitting baseline overdispersion via MLE...")
log_conc_bl = fit_baseline_overdispersion(raw_train, lib_train)

# update nll_bl in place — all downstream cells use the corrected baseline
nll_bl = eval_baseline_nll(log_conc_bl, raw_val, lib_val)
total_nll_bl = float(nll_bl.mean())

print(f"\nbaseline NLL  (unc. overdispersion — wrong):   {total_nll_bl_wrong:.4f}")
print(f"baseline NLL  (MLE overdispersion — correct):  {total_nll_bl:.4f}")
print(
    f"confound magnitude:                            {total_nll_bl - total_nll_bl_wrong:+.4f}"
)
print(f"\ncausal NLL:                                    {total_nll_cau:.4f}")
print(
    f"ΔNLL (baseline − causal)  before fix:          {total_nll_bl_wrong - total_nll_cau:+.4f}"
)
print(
    f"ΔNLL (baseline − causal)  after  fix:          {total_nll_bl - total_nll_cau:+.4f}"
)

In [ ]:
nll_unc.shape

## 5. Analysis

### 5.1 Model comparison

**Aggregate val NLL (mean over all genes and cells):**

| Model | NLL |
|---|---|
| Baseline (ctrl mean, MLE overdispersion) | 1.4956 |
| Causal (RegulonDB-masked linear) | 1.6033 |
| Unconstrained (dense linear) | 2.5419 |

**Interpretation:** baseline ≈ causal >> unconstrained. The dense model overfits badly (~1M parameters, 1,591 training cells); causal sparsity acts simultaneously as a biology-informed prior and as regularization. The aggregate NLL improvement of causal over baseline is small because detectable TF signal is concentrated in a minority of genes — the aggregate dilutes the effect.

In [ ]:
from IPython.display import display

has_reg = Amask_tf.sum(1) > 0
REG_COLORS = {"has regulator(s)": "#ff7f0e", "no known regulator": "#d3d3d3"}
NEUTRAL = "#7a7a7a"

# ── training curves ───────────────────────────────────────────────────────────
df_hist = pd.DataFrame(
    {
        "epoch": list(range(len(hist_unc))) + list(range(len(hist_cau))),
        "NLL": hist_unc + hist_cau,
        "model": ["unconstrained"] * len(hist_unc)
        + ["causal (RegulonDB)"] * len(hist_cau),
    }
)
p_train = (
    gg.ggplot(df_hist, gg.aes(x="epoch", y="NLL", color="model"))
    + gg.geom_line(alpha=0.8, size=0.6)
    + gg.scale_color_manual(
        values={"unconstrained": "#1f77b4", "causal (RegulonDB)": "#ff7f0e"}
    )
    + gg.labs(x="epoch", y="train NLL / cell", title="training curves", color="")
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(figure_size=(3, 2), legend_position="bottom")
)

# ── NLL distributions: boxplots ───────────────────────────────────────────────
NLL_CLIP = float(np.percentile(np.concatenate([nll_bl, nll_unc, nll_cau]), 99))
MODEL_ORDER = ["baseline", "causal", "unconstrained"]
df_box = pd.DataFrame(
    {
        "model": pd.Categorical(
            ["baseline"] * n_genes + ["causal"] * n_genes + ["unconstrained"] * n_genes,
            categories=MODEL_ORDER,
            ordered=True,
        ),
        "NLL": np.clip(np.concatenate([nll_bl, nll_cau, nll_unc]), 0, NLL_CLIP),
    }
)
p_box = (
    gg.ggplot(df_box, gg.aes(x="model", y="NLL"))
    + gg.geom_boxplot(
        fill=NEUTRAL, color="black", width=0.5, outlier_alpha=0.25, outlier_size=0.4
    )
    + gg.labs(x="", y="NLL / cell", title="per-gene val NLL distribution")
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(figure_size=(3, 2.5))
)


# ── pairwise NLL scatters (neutral color) ─────────────────────────────────────
def scatter_nll(x_nll, y_nll, x_label, y_label, title):
    pct99 = float(np.percentile(np.concatenate([x_nll, y_nll]), 99))
    vmin = float(min(x_nll.min(), y_nll.min())) - 0.05
    lim = (vmin, pct99 + 0.1)
    df = pd.DataFrame({"x": np.clip(x_nll, *lim), "y": np.clip(y_nll, *lim)})
    return (
        gg.ggplot(df, gg.aes(x="x", y="y"))
        + gg.geom_point(color=NEUTRAL, size=0.5, alpha=0.4)
        + gg.geom_abline(
            slope=1, intercept=0, linetype="dashed", color="black", size=0.4
        )
        + gg.coord_fixed(xlim=lim, ylim=lim)
        + gg.labs(x=x_label, y=y_label, title=title)
        + PLOTNINE_DEFAULT_THEME_2
        + gg.theme(figure_size=(2.5, 2.5))
    )


p_bl_cau = scatter_nll(
    nll_bl,
    nll_cau,
    x_label="baseline NLL",
    y_label="causal NLL",
    title="causal vs baseline",
)
p_unc_cau = scatter_nll(
    nll_unc,
    nll_cau,
    x_label="unconstrained NLL",
    y_label="causal NLL",
    title="causal vs unconstrained",
)

# ── ranked improvement: unconstrained → causal ────────────────────────────────
delta = nll_unc - nll_cau
order = np.argsort(delta)[::-1]
df_bar = pd.DataFrame(
    {
        "rank": np.arange(n_genes),
        "delta": delta[order],
        "regulator": np.where(has_reg[order], "has regulator(s)", "no known regulator"),
    }
)
p_bar = (
    gg.ggplot(df_bar, gg.aes(x="rank", y="delta", fill="regulator"))
    + gg.geom_col(width=1.0)
    + gg.geom_hline(yintercept=0, color="black", size=0.4)
    + gg.scale_fill_manual(values=REG_COLORS)
    + gg.labs(
        x="gene rank",
        y="ΔNLL (unconstrained − causal)\n+ = causal wins",
        title="improvement from causal constraint",
        fill="",
    )
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(figure_size=(5.5, 2.5), legend_position="bottom")
)

fig_train = p_train.draw()
fig_box = p_box.draw()
fig_bl_cau = p_bl_cau.draw()
fig_unc_cau = p_unc_cau.draw()
fig_bar = p_bar.draw()

display(fig_train, fig_box, fig_bl_cau, fig_unc_cau, fig_bar)

### 5.2 Gene-centric analysis

**988 / 4,725 genes (~21%) where causal beats baseline (ΔNLL > 0).** These are the genes where TF expression variation provides signal beyond the control mean — the "specifically regulated" layer of the transcriptome.

Signal is concentrated in a narrow biological regime:
- **ctrl mean log-CP10K ∈ [0, 0.4]:** lowly expressed, non-constitutive genes
- **Q2/Q3 expression quartiles:** regulated middle range; Q1 (silent) is trivially predicted by both, Q4 (constitutive housekeeping) is barely moved by TF knockdowns
- **val std log-CP10K ≈ 0.2:** moderate, condition-specific variability — genes that respond to a specific subset of TFs reproducibly; high-std genes are driven by many regulators or noise, low-std genes are invariant

**These 988 genes form the forward analysis gene set for all downstream gene-centric and TF-centric analyses.**

In [ ]:
delta = nll_unc - nll_cau
order = np.argsort(delta)[::-1]
top_genes = [
    (var_names[i], float(delta[i]), int(has_reg[i]), int(Amask_tf[i].sum()))
    for i in order[:20]
]
df_top = pd.DataFrame(
    top_genes, columns=["gene", "delta_nll", "has_regulator", "n_regulators"]
)
print("Top 20 genes where causal model wins:")
print(df_top.to_string(index=False))

In [ ]:
from IPython.display import display

# ── gene-level summary ────────────────────────────────────────────────────────
delta_vs_bl = nll_bl - nll_cau
ctrl_lcp_std = lcp_train[ctrl_mask].std(0)
val_lcp_std = lcp_val.std(0)
n_regs = Amask_tf.sum(1).astype(int)

df_gene = pd.DataFrame(
    {
        "gene": var_names,
        "ctrl_mean": ctrl_lcp_mean,
        "ctrl_std": ctrl_lcp_std,
        "val_std": val_lcp_std,
        "nll_bl": nll_bl,
        "nll_cau": nll_cau,
        "delta_vs_bl": delta_vs_bl,
        "has_reg": has_reg,
        "n_regs": n_regs,
        "reg_label": np.where(has_reg, "has regulator(s)", "no known regulator"),
    }
)
df_gene["outcome"] = pd.Categorical(
    np.where(delta_vs_bl > 0, "causal wins", "baseline wins"),
    categories=["baseline wins", "causal wins"],
    ordered=True,
)
OUTCOME_COLORS = {"causal wins": "#ff7f0e", "baseline wins": "#4878d0"}

DELTA_CLIP = float(np.percentile(np.abs(delta_vs_bl), 99.5))

# extreme genes for fig 2 labels (top/bottom 10 by ΔNLL)
df_labels = pd.concat(
    [
        df_gene.nlargest(10, "delta_vs_bl"),
        df_gene.nsmallest(10, "delta_vs_bl"),
    ]
).assign(delta_clipped=lambda d: np.clip(d["delta_vs_bl"], -DELTA_CLIP, DELTA_CLIP))
df_b = df_gene.assign(delta_clipped=np.clip(delta_vs_bl, -DELTA_CLIP, DELTA_CLIP))

# ── fig 2: ΔNLL vs ctrl_mean — color by outcome, label extremes ──────────────
# improvement at low expression → causal wins for lowly-expressed genes.
# baseline-wins mass visible at the expression levels where it concentrates.

p2 = (
    gg.ggplot(
        df_b.sort_values("outcome"),
        gg.aes(x="ctrl_mean", y="delta_clipped", color="outcome"),
    )
    + gg.geom_point(alpha=0.35, size=0.5)
    + gg.geom_hline(yintercept=0, linetype="dashed", color="black", size=0.4)
    + gg.geom_text(
        gg.aes(x="ctrl_mean", y="delta_clipped", label="gene"),
        data=df_labels,
        inherit_aes=False,
        size=5,
        nudge_y=0.12,
        color="black",
    )
    + gg.scale_color_manual(values=OUTCOME_COLORS)
    + gg.labs(
        x="mean log-CP10K (ctrl)",
        y="ΔNLL (baseline − causal)\n+ = causal wins",
        title="improvement vs ctrl expression level",
        color="",
    )
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(figure_size=(4.5, 3.5), legend_position="bottom")
)

# ── fig 3: fraction of outcomes by expression quartile ────────────────────────
# stacked bar directly answers: at what expression levels does baseline > causal?
# if "baseline wins" accumulates in Q3/Q4, improvement is a low-expression phenomenon.

df_gene["expr_q"] = pd.qcut(
    df_gene["ctrl_mean"],
    q=4,
    labels=["Q1 (low)", "Q2", "Q3", "Q4 (high)"],
)
df_prop = (
    df_gene.groupby(["expr_q", "outcome"], observed=True).size().reset_index(name="n")
)
df_prop["frac"] = df_prop.groupby("expr_q", observed=True)["n"].transform(
    lambda x: x / x.sum()
)

p3 = (
    gg.ggplot(df_prop, gg.aes(x="expr_q", y="frac", fill="outcome"))
    + gg.geom_col(position="stack", width=0.65)
    + gg.geom_hline(yintercept=0.5, linetype="dashed", color="white", size=0.5)
    + gg.scale_fill_manual(values=OUTCOME_COLORS)
    + gg.labs(
        x="ctrl mean expression quartile",
        y="fraction of genes",
        title="outcome by expression level",
        fill="",
    )
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(figure_size=(3.5, 2.5), legend_position="bottom")
)

# ── fig 4: val_std density conditioned on outcome ─────────────────────────────
# observation 4: low val_std → causal wins.
# if densities separate: causal-wins genes have tighter val distribution,
# meaning these are genes that are near-silent in rich medium and sharply
# induced in a subset of conditions — exactly where TF edges should matter.

p4 = (
    gg.ggplot(df_gene, gg.aes(x="val_std", color="outcome", fill="outcome"))
    + gg.geom_density(alpha=0.35, size=0.6)
    + gg.scale_color_manual(values=OUTCOME_COLORS)
    + gg.scale_fill_manual(values=OUTCOME_COLORS)
    + gg.labs(
        x="std log-CP10K (val conditions)",
        y="density",
        title="conditional variability by outcome",
        color="",
        fill="",
    )
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(figure_size=(3.5, 2.5), legend_position="bottom")
)

fig2 = p2.draw()
fig3 = p3.draw()
fig4 = p4.draw()

display(fig2, fig3, fig4)

In [ ]:
# genes where causal beats baseline, sorted by improvement
df_wins = (
    df_gene[df_gene["delta_vs_bl"] > 0]
    .sort_values("delta_vs_bl", ascending=False)[
        ["gene", "delta_vs_bl", "ctrl_mean", "val_std", "n_regs", "has_reg"]
    ]
    .reset_index(drop=True)
)

# attach regulator names from RegulonDB
reg_map = (
    ref_db[ref_db["target_gene"].str.lower().isin(df_wins["gene"])]
    .groupby("target_gene")["regulator_gene"]
    .apply(lambda x: ", ".join(sorted(set(x.str.lower()))))
    .rename("regulators")
)
df_wins = df_wins.merge(
    reg_map.reset_index().rename(columns={"target_gene": "gene"}),
    on="gene",
    how="left",
)
df_wins["regulators"] = df_wins["regulators"].fillna("—")

print(f"{len(df_wins)} genes where causal wins (ΔNLL > 0)\n")
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.3f}".format)
print(df_wins.to_string(index=False))

### 5.3 TF-centric analysis

**183 / 217 TF genes have ≥ 1 target where causal beats baseline.** Only 34 TFs never contribute — likely because all their RegulonDB targets fall in the Q1 (silent) or Q4 (constitutive) regime where baseline already wins by default. This is not evidence those TFs are inactive; it means their targets lie outside the detectable expression window in this dataset.

In [ ]:
from IPython.display import display

# ── per-TF table ──────────────────────────────────────────────────────────────
records = []
for k, tf_name in enumerate(tf_genes):
    t_idx = np.where(Amask_tf[:, k] > 0)[0]
    if len(t_idx) == 0:
        continue
    n_targets = len(t_idx)
    n_wins = int(np.sum(delta_vs_bl[t_idx] > 0))
    records.append(
        {
            "TF": tf_name,
            "n_wins": n_wins,
            "n_targets": n_targets,
            "pct": 100.0 * n_wins / n_targets,
        }
    )

df_tf_wins = (
    pd.DataFrame(records).sort_values("pct", ascending=False).reset_index(drop=True)
)

print(
    df_tf_wins[df_tf_wins["n_wins"] > 0]
    .rename(columns={"n_wins": "#", "n_targets": "targets", "pct": "%"})[
        ["TF", "#", "%"]
    ]
    .to_string(index=False, float_format="{:.1f}".format)
)

# ── histogram of % with few bins ─────────────────────────────────────────────
p_hist = (
    gg.ggplot(df_tf_wins, gg.aes(x="pct"))
    + gg.geom_histogram(bins=10, fill=NEUTRAL, color="white", size=0.4)
    + gg.labs(x="% of targets where causal wins", y="# TFs")
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(figure_size=(3.5, 2.5))
)

fig_hist = p_hist.draw()
display(fig_hist)

### 5.4 Perturbation-centric analysis

For each of the 4,050 non-TF perturbations in the val set, count the number of genes where causal NLL < baseline NLL. This measures how much TF-regulatory signal each perturbation engages.

In [ ]:
# ── per-perturbation causal-wins count ───────────────────────────────────────
# Process all val cells in batches; scatter-add NLL into per-perturbation accumulators.

pert_labels = np.array(adata_val.obs[PERT_COL])
unique_perts, pert_idx = np.unique(pert_labels, return_inverse=True)
n_perts = len(unique_perts)

nll_cau_sum = np.zeros((n_perts, n_genes), dtype=np.float64)
nll_bl_sum = np.zeros((n_perts, n_genes), dtype=np.float64)
pert_counts = np.zeros(n_perts, dtype=np.int32)

W_masked = np.array(state_cau.params["W"]) * Amask_tf  # (n_genes, n_tfs)
b_cau = np.array(state_cau.params["b"])  # (n_genes,)
conc_cau = jnp.exp(state_cau.params["overdispersion_"])
mu_bl = jnp.maximum(jnp.expm1(jnp.array(ctrl_lcp_mean)), 1e-8)
conc_bl = jnp.exp(jnp.array(log_conc_bl))

_W = jnp.array(W_masked)
_b = jnp.array(b_cau)
_xm = jnp.array(ctrl_lcp_mean)


@jax.jit
def batch_nll(x_b, y_b, l_b):
    lcp = _xm + x_b @ _W.T + _b
    mean_c = jnp.maximum(jnp.expm1(lcp) * l_b[:, None] / 1e4, 1e-8)
    lp_c = NegativeBinomial2(mean=mean_c, concentration=conc_cau).log_prob(y_b)

    mean_b = mu_bl[None, :] * l_b[:, None] / 1e4
    lp_b = NegativeBinomial2(mean=mean_b, concentration=conc_bl).log_prob(y_b)

    return -lp_c, -lp_b  # (B, n_genes) each, positive NLL


BATCH = 512
n_val = len(Xval)
for s in tqdm(range(0, n_val, BATCH), desc="per-pert NLL"):
    sl = slice(s, s + BATCH)
    x_b = jnp.array(Xval[sl])
    y_b = jnp.array(raw_val[sl])
    l_b = jnp.array(lib_val[sl])
    pi = pert_idx[sl]

    nc, nb = batch_nll(x_b, y_b, l_b)
    np.add.at(nll_cau_sum, pi, np.array(nc))
    np.add.at(nll_bl_sum, pi, np.array(nb))
    np.add.at(pert_counts, pi, 1)

nll_cau_pert = nll_cau_sum / pert_counts[:, None]  # (n_perts, n_genes)
nll_bl_pert = nll_bl_sum / pert_counts[:, None]

delta_pert = nll_bl_pert - nll_cau_pert  # + = causal wins
n_causal_wins = (delta_pert > 0).sum(1)  # (n_perts,)
frac_causal_wins = n_causal_wins / n_genes

df_pert = pd.DataFrame(
    {
        "perturbation": unique_perts,
        "n_cells": pert_counts,
        "n_causal_wins": n_causal_wins,
        "frac_causal_wins": frac_causal_wins,
    }
)

print(f"Perturbations: {n_perts}")
print(
    f"Cells/pert  — median: {np.median(pert_counts):.0f}  min: {pert_counts.min()}  max: {pert_counts.max()}"
)
print(
    f"\nn_causal_wins — mean: {n_causal_wins.mean():.1f}  median: {np.median(n_causal_wins):.0f}"
    f"  p5: {np.percentile(n_causal_wins, 5):.0f}  p95: {np.percentile(n_causal_wins, 95):.0f}"
)
df_pert.head()

In [ ]:
from IPython.display import display

aggregate_wins = int(
    (nll_bl - nll_cau > 0).sum()
)  # 988 — aggregate across all val cells

# ── histogram: n_causal_wins per perturbation ─────────────────────────────────
p_wins_hist = (
    gg.ggplot(df_pert, gg.aes(x="n_causal_wins"))
    + gg.geom_histogram(bins=50, fill=NEUTRAL, color="white", size=0.3)
    + gg.geom_vline(
        xintercept=aggregate_wins, linetype="dashed", color="black", size=0.6
    )
    + gg.annotate(
        "text",
        x=aggregate_wins + 15,
        y=0,
        label=f"aggregate ({aggregate_wins})",
        ha="left",
        va="bottom",
        size=7,
        color="black",
    )
    + gg.labs(
        x="# genes where causal > baseline",
        y="# perturbations",
        title="per-perturbation causal wins (all 4,725 genes)",
    )
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(figure_size=(4.5, 3))
)

# ── scatter: n_causal_wins vs n_cells (reliability check) ────────────────────
p_wins_cells = (
    gg.ggplot(df_pert, gg.aes(x="n_cells", y="n_causal_wins"))
    + gg.geom_point(color=NEUTRAL, size=0.6, alpha=0.4)
    + gg.geom_hline(
        yintercept=aggregate_wins, linetype="dashed", color="black", size=0.4
    )
    + gg.labs(
        x="# cells for perturbation",
        y="# genes where causal wins",
        title="wins vs cell count",
    )
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(figure_size=(3.5, 3))
)

fig_wins_hist = p_wins_hist.draw()
fig_wins_cells = p_wins_cells.draw()
display(fig_wins_hist, fig_wins_cells)

In [ ]:
from IPython.display import display

# Each point is a gene.
# Both axes use equal-perturbation weighting (each of the 4,050 perturbations counts once,
# regardless of cell count) so x and y are consistent with each other.
# Color uses the original cell-weighted aggregate (the 988-gene result from Section 5.2).
#
# x: mean per-perturbation ΔNLL  (baseline − causal, averaged equally over perturbations)
# y: fraction of perturbations where causal is worse than baseline for this gene
#    0 = causal wins in every perturbation  |  1 = causal loses in every perturbation
#
# Because x and y use the same weighting, they are monotonically related:
# off-diagonals between x/y are minimal.  Off-diagonals between *color* and *position*
# mark genes where cell-weighted vs equal-weighted aggregation disagree.

mean_delta_pert = delta_pert.mean(0)  # (n_genes,) equal-pert-weighted mean ΔNLL
frac_causal_worse = (delta_pert < 0).mean(0)  # (n_genes,) equal-pert-weighted fraction

causal_wins_agg = delta_vs_bl > 0  # color: original cell-weighted classification

CLIP = float(np.percentile(np.abs(mean_delta_pert), 99))
df_scatter = pd.DataFrame(
    {
        "gene": var_names,
        "mean_delta_pert": np.clip(mean_delta_pert, -CLIP, CLIP),
        "frac_causal_worse": frac_causal_worse,
        "avg_outcome": pd.Categorical(
            np.where(
                causal_wins_agg,
                "causal wins on average (cell-weighted)",
                "baseline wins on average (cell-weighted)",
            ),
            categories=[
                "baseline wins on average (cell-weighted)",
                "causal wins on average (cell-weighted)",
            ],
            ordered=True,
        ),
    }
)

p_scatter = (
    gg.ggplot(
        df_scatter.sort_values("avg_outcome"),
        gg.aes(x="mean_delta_pert", y="frac_causal_worse", color="avg_outcome"),
    )
    + gg.geom_point(size=0.5, alpha=0.35)
    # reference lines
    + gg.geom_vline(xintercept=0, linetype="dashed", color="black", size=0.4)
    + gg.geom_hline(yintercept=0.5, linetype="dotted", color="#777777", size=0.5)
    # label the 0.5 line
    + gg.annotate(
        "text",
        x=-CLIP * 0.97,
        y=0.52,
        label="causal wins in 50% of perturbations",
        ha="left",
        va="bottom",
        size=6.5,
        color="#555555",
    )
    # y-axis pole labels
    + gg.annotate(
        "text",
        x=-CLIP * 0.97,
        y=0.01,
        label="▲ causal wins in every perturbation",
        ha="left",
        va="bottom",
        size=6.5,
        color="#333333",
    )
    + gg.annotate(
        "text",
        x=-CLIP * 0.97,
        y=0.99,
        label="▼ causal loses in every perturbation",
        ha="left",
        va="top",
        size=6.5,
        color="#333333",
    )
    # quadrant labels (orange side only)
    + gg.annotate(
        "text",
        x=CLIP * 0.7,
        y=0.08,
        label="robustly\npredictable",
        ha="center",
        va="bottom",
        size=7,
        color="#cc5500",
    )
    + gg.annotate(
        "text",
        x=CLIP * 0.7,
        y=0.92,
        label="aggregate signal,\nnoisy per perturbation",
        ha="center",
        va="top",
        size=7,
        color="#cc5500",
    )
    + gg.scale_y_continuous(breaks=[0, 0.25, 0.5, 0.75, 1.0])
    + gg.scale_color_manual(
        values={
            "causal wins on average (cell-weighted)": "#ff7f0e",
            "baseline wins on average (cell-weighted)": "#4878d0",
        }
    )
    + gg.labs(
        x="mean per-perturbation ΔNLL  (baseline − causal, equal-perturbation weighting)",
        y="fraction of perturbations where\ncausal is worse than baseline",
        title="gene-level causal predictability: per-perturbation view",
        color="cell-weighted aggregate (Section 5.2):",
    )
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(figure_size=(5.5, 4.5), legend_position="bottom")
)

display(p_scatter.draw())

#### 5.4.1 Metabolic perturbations: do they disrupt TF-regulatory predictability?

**Hypothesis:** knocking out a metabolic enzyme changes metabolite concentrations, which can alter TF *activity* post-translationally (allosteric regulation, phosphorylation, cofactor availability) without changing TF *mRNA levels*. Since the causal model inputs TF expression, such perturbations should cause specific model failures on TF-regulated target genes.

**Analysis:** for each gene, compare  
- **x-axis** — *median* ΔNLL across all 4,050 perturbations: quantile-based measure of robust overall predictability  
- **y-axis** — *mean* ΔNLL across metabolic enzyme KDs only: metabolic-specific predictability  

**Bottom-right quadrant** (x > 0, y < 0): well-predicted on average, but specifically fails in metabolic KDs → candidate genes whose regulating TF has activity coupled to metabolism.  
**Diagonal** (y = x): metabolic KDs behave identically to the average perturbation; points below the diagonal are specifically worse in metabolic context.

In [ ]:
import json
import plotly.express as px

FBA_JSON = "/workspace/data/05142026_metabolome/iJO1366.json"
with open(FBA_JSON) as f:
    fba_data = json.load(f)
metabolic_genes = {g["name"].lower() for g in fba_data["genes"]}

is_metabolic = np.array([p in metabolic_genes for p in unique_perts])
n_metabolic = int(is_metabolic.sum())
n_metabolic_cells = int(pert_counts[is_metabolic].sum())
print(
    f"{n_metabolic} metabolic enzyme KDs  |  {n_metabolic_cells} cells  |  "
    f"median {int(np.median(pert_counts[is_metabolic]))} cells/KD"
)

# ── per-gene stats ─────────────────────────────────────────────────────────────
median_delta_all = np.median(delta_pert, axis=0)  # (n_genes,) robust aggregate
mean_delta_metabolic = delta_pert[is_metabolic].mean(0)  # (n_genes,) metabolic-specific

# shared clip so the diagonal y = x has slope 1
CLIP = max(
    float(np.percentile(np.abs(median_delta_all), 99.5)),
    float(np.percentile(np.abs(mean_delta_metabolic), 99.5)),
)

df_met = pd.DataFrame(
    {
        "gene": var_names,
        "median_delta_all": np.clip(median_delta_all, -CLIP, CLIP),
        "mean_delta_metabolic": np.clip(mean_delta_metabolic, -CLIP, CLIP),
        "K_Y": np.where(delta_vs_bl > 0, "causal wins", "baseline wins"),
    }
).sort_values(
    "K_Y"
)  # baseline wins plotted first → causal wins on top

# ── plotly scatter ─────────────────────────────────────────────────────────────
fig = px.scatter(
    df_met,
    x="median_delta_all",
    y="mean_delta_metabolic",
    color="K_Y",
    hover_name="gene",
    color_discrete_map={"causal wins": "#ff7f0e", "baseline wins": "#4878d0"},
    category_orders={"K_Y": ["baseline wins", "causal wins"]},
    labels={
        "median_delta_all": "median ΔNLL over all perturbations",
        "mean_delta_metabolic": f"mean ΔNLL over metabolic KDs (n={n_metabolic})",
        "K_Y": "",
    },
    title="metabolic KDs: overall vs metabolic-specific causal predictability",
    opacity=0.5,
)
fig.update_traces(marker_size=4)
fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)
fig.add_vline(x=0, line_dash="dash", line_color="black", line_width=1)
fig.add_shape(
    type="line",
    x0=-CLIP,
    y0=-CLIP,
    x1=CLIP,
    y1=CLIP,
    line=dict(dash="dot", color="#999999", width=1),
)
fig.add_annotation(
    x=CLIP * 0.55,
    y=-CLIP * 0.88,
    text="well predicted overall<br>but fails in metabolic KDs",
    showarrow=False,
    font=dict(size=11, color="#cc5500"),
)
fig.update_layout(
    width=700,
    height=550,
    legend=dict(orientation="h", yanchor="top", y=-0.12),
)
fig.show()

# ── candidate table ────────────────────────────────────────────────────────────
mask_interesting = (
    (median_delta_all > 0) & (mean_delta_metabolic < 0) & (delta_vs_bl > 0)
)
df_candidates = (
    pd.DataFrame(
        {
            "gene": var_names,
            "median_delta_all": median_delta_all,
            "mean_delta_metabolic": mean_delta_metabolic,
            "specificity": median_delta_all - mean_delta_metabolic,
            "has_reg": has_reg,
        }
    )
    .loc[mask_interesting]
    .sort_values("specificity", ascending=False)
    .head(20)
)
print(f"\nTop 20 K_Y genes most specifically disrupted in metabolic KDs:")
print(df_candidates.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
def rank_perturbations(gene_name, ascending=True):
    """
    Rank all perturbations by ΔNLL for a given gene.

    ascending=True  → most negative first (perturbations where causal fails most)
    ascending=False → most positive first (perturbations where causal wins most)

    Returns a DataFrame with columns:
      perturbation, delta_nll, n_cells, is_metabolic
    """
    g = gene_idx[gene_name.lower()]
    delta_g = delta_pert[:, g]
    order = np.argsort(delta_g) if ascending else np.argsort(delta_g)[::-1]
    return pd.DataFrame(
        {
            "perturbation": unique_perts[order],
            "delta_nll": delta_g[order],
            "n_cells": pert_counts[order],
            "is_metabolic": is_metabolic[order],
        }
    ).reset_index(drop=True)


# ── example: phoa is the top K_Y gene overall ─────────────────────────────────
display(rank_perturbations("phoa").head(20))

In [ ]:
from scipy.stats import median_abs_deviation as mad

# ── restrict to the 988 K_Y genes (causal beats baseline in aggregate) ────────
reg_mask = delta_vs_bl > 0  # (n_genes,) bool, 988 genes
delta_reg = delta_pert[:, reg_mask]  # (n_perts, n_ky_genes)
reg_names = [var_names[i] for i in np.where(reg_mask)[0]]

# ── robust per-gene Gaussian fit ──────────────────────────────────────────────
mu_robust = np.median(delta_reg, axis=0)  # (n_ky_genes,)
sigma_robust = mad(delta_reg, axis=0, scale="normal")  # (n_ky_genes,)

# regularize σ: shrink toward global noise floor in quadrature
# prevents near-zero per-gene MAD from producing spuriously large z-scores
sigma0 = float(np.median(sigma_robust))  # global median σ across K_Y genes
sigma_reg = np.sqrt(sigma_robust**2 + sigma0**2)  # (n_ky_genes,)

# ── z-scores for enzyme perturbations only ────────────────────────────────────
delta_enz = delta_pert[is_metabolic][:, reg_mask]  # (n_enz, n_ky_genes)
z_enz = (delta_enz - mu_robust[None, :]) / sigma_reg[None, :]

enz_names = unique_perts[is_metabolic]

df_z = (
    pd.DataFrame(
        {
            "gene_name": np.tile(reg_names, len(enz_names)),
            "perturbation_name": np.repeat(enz_names, len(reg_names)),
            "z_score": z_enz.ravel(),
        }
    )
    .sort_values("z_score")
    .reset_index(drop=True)
)

print(
    f"df_z shape: {df_z.shape}  ({len(enz_names)} enzyme KDs × {len(reg_names)} K_Y genes)"
)
print(f"global σ₀ = {sigma0:.4f}  (regularization floor)")
print(f"\nmost negative z-scores (causal specifically fails in these enzyme KDs):")
display(df_z.head(30))

In [ ]:
from IPython.display import display

# per-gene summary: most extreme disruption across all metabolic enzyme KDs
min_z_per_gene = z_enz.min(0)   # (n_ky_genes,) most negative z any enzyme achieves

df_gene_z = pd.DataFrame(
    {
        "gene":      reg_names,
        "mu_robust": mu_robust,
        "min_z":     min_z_per_gene,
        "reg_label": pd.Categorical(
            np.where(
                has_reg[reg_mask],
                "has known TF regulator",
                "no known regulator",
            ),
            categories=["no known regulator", "has known TF regulator"],
            ordered=True,
        ),
    }
).sort_values("reg_label")  # no-regulator plotted first → TF-regulated on top

x_max = float(df_gene_z["mu_robust"].quantile(0.99))
x_min = float(df_gene_z["mu_robust"].quantile(0.01))

p_z = (
    gg.ggplot(df_gene_z, gg.aes(x="mu_robust", y="min_z", color="reg_label"))
    + gg.geom_point(size=0.8, alpha=0.5)
    + gg.geom_hline(yintercept=0,  linetype="dashed", color="black",   size=0.4)
    + gg.geom_hline(yintercept=-2, linetype="dotted", color="#777777", size=0.5)
    + gg.geom_hline(yintercept=-3, linetype="dotted", color="#cc5500", size=0.5)
    + gg.annotate(
        "text", x=x_max, y=-1.75,
        label="z = −2", ha="right", va="bottom", size=6.5, color="#777777",
    )
    + gg.annotate(
        "text", x=x_max, y=-2.75,
        label="z = −3", ha="right", va="bottom", size=6.5, color="#cc5500",
    )
    + gg.annotate(
        "text", x=x_max, y=0.15,
        label="causal = baseline (on average)", ha="right", va="bottom", size=6, color="#555555",
    )
    + gg.scale_color_manual(
        values={
            "has known TF regulator": "#ff7f0e",
            "no known regulator":     "#4878d0",
        }
    )
    + gg.labs(
        x="median ΔNLL across all perturbations  (gene-level predictability)",
        y="min z-score across metabolic enzyme KDs\n(most extreme disruption)",
        title="metabolic disruption specificity — K_Y genes",
        color="",
    )
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(figure_size=(5.5, 4.5), legend_position="bottom")
)
display(p_z.draw())

n_disrupted    = int((min_z_per_gene < -2).sum())
n_disrupted_tf = int(((min_z_per_gene < -2) & has_reg[reg_mask]).sum())
print(f"\nK_Y genes with min z < −2: {n_disrupted}  (of which with TF regulator: {n_disrupted_tf})")

In [ ]:
bins = np.linspace(-5, 5, 51)
df_z["z_score"].hist(bins=bins)